# T2 · Snowflake It, Then Justify It
Subtasks: Part A (DimProvince), Part B (DimCountry), Part C (DimCity normalized), checks, Part D (verdict). All in this one notebook.

In [1]:
import pandas as pd, os
from collections import Counter
pd.set_option("display.max_colwidth", None, "display.width", 250)

d = next(x for x in [".", "..", "/mnt/user-data/uploads"] if os.path.exists(os.path.join(x, "FactShipment.csv")))
rd = lambda n: pd.read_csv(os.path.join(d, n + ".csv"))
Fact, flat = rd("FactShipment"), rd("DimCity")
assert Fact.WeightKG.sum() == 578_650 and Fact.FreightCostPKR.sum() == 4_808_000
cities = flat[flat.CityKey != -1]
flat[["CityKey", "CityName", "Province", "Region"]]

,CityKey,CityName,Province,Region
0,1,Karachi,Sindh,South
1,2,Gwadar,Balochistan,South
2,3,Lahore,Punjab,North
3,4,Multan,Punjab,Central
4,5,Islamabad,Islamabad ICT,North
5,6,Quetta,Balochistan,West
6,7,Faisalabad,Punjab,Central
7,8,Rawalpindi,Punjab,North
8,9,Sialkot,Punjab,North
9,10,Peshawar,Khyber Pakhtunkhwa,North


## Part A · DimProvince

In [2]:
# One row per distinct province (first-appearance order), keys minted here. Region moves across from DimCity.
provinces = list(dict.fromkeys(cities.Province))

def region_of(p):                       # a province must carry ONE region: take the most common, ties to first appearance
    return Counter(cities.loc[cities.Province == p, "Region"]).most_common(1)[0][0]

DimProvince = pd.DataFrame({"ProvinceKey": range(1, len(provinces) + 1), "ProvinceName": provinces,
                            "RegionName": [region_of(p) for p in provinces], "CountryKey": 1})
DimProvince = pd.concat([DimProvince, pd.DataFrame([{"ProvinceKey": -1, "ProvinceName": "Unknown", "RegionName": "Unknown", "CountryKey": -1}])],
                        ignore_index=True)
print("Rows in DimProvince, excluding Unknown:", int((DimProvince.ProvinceKey != -1).sum()))
DimProvince

Rows in DimProvince, excluding Unknown: 5


,ProvinceKey,ProvinceName,RegionName,CountryKey
0,1,Sindh,South,1
1,2,Balochistan,South,1
2,3,Punjab,North,1
3,4,Islamabad ICT,North,1
4,5,Khyber Pakhtunkhwa,North,1
5,-1,Unknown,Unknown,-1


## Part B · DimCountry

In [3]:
DimCountry = pd.DataFrame([(1, "Pakistan", "South Asia"), (-1, "Unknown", "Unknown")], columns=["CountryKey", "CountryName", "RegionGroup"])
country_rows = int((DimCountry.CountryKey != -1).sum())
print("Rows in DimCountry, excluding Unknown:", country_rows)
DimCountry

Rows in DimCountry, excluding Unknown: 1


,CountryKey,CountryName,RegionGroup
0,1,Pakistan,South Asia
1,-1,Unknown,Unknown


## Part C · DimCity, normalized (Province and Region removed, ProvinceKey added)

In [4]:
pkey = DimProvince.set_index("ProvinceName").ProvinceKey
DimCity = flat.assign(ProvinceKey=flat.Province.map(pkey))[["CityKey", "CityName", "ProvinceKey", "IsPortCity", "FacilityType"]]
assert len(DimCity) == 15 and DimCity.ProvinceKey.notna().all()
DimCity['ProvinceKey'] = DimCity.ProvinceKey.astype(int)
DimCity

,CityKey,CityName,ProvinceKey,IsPortCity,FacilityType
0,1,Karachi,1,Yes,Sea Port + Rail Hub
1,2,Gwadar,2,Yes,Sea Port
2,3,Lahore,3,No,Dry Port
3,4,Multan,3,No,Inland Depot
4,5,Islamabad,4,No,Dry Port
5,6,Quetta,2,No,Inland Depot
6,7,Faisalabad,3,No,Dry Port
7,8,Rawalpindi,3,No,Rail Hub
8,9,Sialkot,3,No,Dry Port
9,10,Peshawar,5,No,Inland Depot


## Checks

In [5]:
# 1. The tables I built match the files supplied with the task
for mine, name in [(DimProvince, "DimProvince"), (DimCountry, "DimCountry"), (DimCity, "DimCity_Snowflaked")]:
    assert mine.reset_index(drop=True).equals(rd(name)), name + " differs from the supplied file"

# 2. Referential integrity: no orphan keys
assert DimCity.ProvinceKey.isin(DimProvince.ProvinceKey).all() and DimProvince.CountryKey.isin(DimCountry.CountryKey).all()

# 3. Totals must not move, and the chain must not fan out the fact rows
chain = (Fact.merge(DimCity, left_on="OriginCityKey", right_on="CityKey", how="left", validate="m:1")
             .merge(DimProvince, on="ProvinceKey", how="left", validate="m:1")
             .merge(DimCountry, on="CountryKey", how="left", validate="m:1"))
assert len(chain) == 34 and chain.WeightKG.sum() == 578_650 and chain.FreightCostPKR.sum() == 4_808_000
flat_w = Fact.merge(flat, left_on="OriginCityKey", right_on="CityKey").groupby("Province").WeightKG.sum()
assert chain.groupby("ProvinceName").WeightKG.sum().sort_index().equals(flat_w.sort_index().rename_axis("ProvinceName"))
print("34 rows | 578,650 kg | PKR 4,808,000 | province weights identical to the flat model")

34 rows | 578,650 kg | PKR 4,808,000 | province weights identical to the flat model


In [6]:
# 4. What did NOT survive the split: a city's Region, now read through its province
cmp = cities[["CityName", "Province", "Region"]].merge(DimCity[["CityName", "ProvinceKey"]], on="CityName").merge(
        DimProvince[["ProvinceKey", "RegionName"]], on="ProvinceKey")
lost = cmp[cmp.Region != cmp.RegionName][["CityName", "Province", "Region", "RegionName"]].rename(
        columns={"Region": "Region in flat DimCity", "RegionName": "Region after snowflaking"})
print("Cities whose Region changes after the split:", len(lost), "of", len(cities))
lost

Cities whose Region changes after the split: 4 of 14


,CityName,Province,Region in flat DimCity,Region after snowflaking
3,Multan,Punjab,Central,North
5,Quetta,Balochistan,West,South
6,Faisalabad,Punjab,Central,North
11,Sukkur,Sindh,Central,South


## Part D · The verdict

**Verdict: Flatten. PakFreight should keep `Province` and `Region` on `DimCity` and not snowflake.**

1. **`DimCountry` has 1 row** (Part B), and only 1 of the 8 T1 questions (Q4, "by country") ever reads it. Snowflaking creates a table, a key, a relationship and a join to store the one word "Pakistan" once; no filter on it can ever select anything different.
2. **T1: the snowflake costs 5 extra joins** (16 against 11) across the eight questions and changes the path of 4 of them (Q3, Q4, Q5, Q8). Every province question the business actually asks (Q3, Q5, Q8) pays one more hop, and none of the 8 questions gets simpler. The extra joins are small lookups, so this is a simplicity and maintenance cost, not a measured speed loss.
3. **The split is not lossless for Region** (check 4 above): in the flat model Region varies *within* a province, so it is not an attribute of the province. Moving it to `DimProvince` forces one region per province, and **4 of 14 cities change Region** (Sukkur, Quetta, Multan, Faisalabad). No total moves (578,650 kg still reconciles), so none of the reconciliation checks would warn about it. To snowflake without that loss, Region would have to stay on `DimCity`, which removes half of what the split was meant to move.
4. **There is nothing large enough to normalize:** `DimCity` has 15 rows and 5 provinces, so the repeated text is 15 short strings, and `DimProvince` would add 6 rows of its own to hold it (Part A; the storage arithmetic is T3).

**What would change my answer:** a second country, province-level attributes that grow beyond a name and a region, or other fact tables at province grain that would share `DimProvince`. None exists in this model today.